In [13]:
from curl_cffi import requests
import pandas as pd
from bs4 import BeautifulSoup

In [14]:
url = "https://portal.iitb.ac.in/asc/Courses"
b_url = "https://portal.iitb.ac.in/asc/"

In [15]:
# YEAR = 2026
# SEMESTER = 1

In [16]:
session = requests.Session()

In [17]:
headers = {
    "Referer": url,
    "Origin": "https://portal.iitb.ac.in/asc"
}

In [18]:
# response = session.get(url)
response = session.get(url, timeout=20)
print(response.status_code)

200


In [19]:
response.status_code

200

In [ ]:
# response.text

In [ ]:
soup = BeautifulSoup(response.content, "html.parser")

In [ ]:
soup

In [ ]:
# for a in soup.select():
departments = []
for d in soup.find_all("a", attrs={"onclick": True}):
    name = d.get_text()
    codes = d["onclick"].split("'")[1].split(",")[:-1]
    for c in codes:
        departments.append((c, name))
        print((c, name))
# print(departments)

In [ ]:
courses = []
for code, name in departments:
    r1 = session.post(url, data = {"deptcd": code, "deptname": name}, headers = headers)
    # print(r1.text)
    s1 = BeautifulSoup(r1.text, "html.parser")
    # print(s1)
    for f in s1.find_all("frame", attrs = {"name": "content"}):
        # print(f)
        src = f.get("src")
        # print(src)
        f_url = b_url + src
        # print(f_url)
        r2 = session.post(f_url, headers = headers)
        # print(r2.text)
        s2 = BeautifulSoup(r2.text, "html.parser")
        # print(s2.prettify())
        for table_row in s2.find_all("tr"):
            present = table_row.find("a", attrs = {"title":"Click Here to view Course Details"})
            # print(present)
            if present:
                # print(present.get_text())
                # course_code = present.get_text()
                cols = table_row.find_all("td")
                # print(cols)
                # print(cols[3].get_text())
                # course_code = cols[1].get_text(strip = True)
                # course_name = cols[2].get_text(strip = True)
                # instructor = cols[3].get_text(strip = True)
                # print(course_code, course_name, instructor)
                courses.append({
                    "dept_code": code,
                    "course_code": cols[1].get_text(strip = True),
                    "course_name": cols[2].get_text(strip = True),
                    "instructor": cols[3].get_text(strip = True),
                })
# print(courses)

In [ ]:
df1 = pd.DataFrame(departments)
df1.to_csv("departments.csv", index = False)
df2 = pd.DataFrame(courses)
df2.to_csv("courses.csv", index = False)